# IMDb Sentiment Analysis Pipeline
### Supervised Binary Classification using TF-IDF and Logistic Regression
This notebook explores the IMDb 50k movie reviews dataset, performs text normalization, feature extraction via unigram/bigram TF-IDF, trains a Logistic Regression classifier, and analyzes the most informative positive and negative features.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Add parent directory for shared modules
sys.path.insert(0, os.path.abspath('..'))
from scripts.preprocess import clean_text
sns.set_theme(style='whitegrid')

## 1. Load Dataset & Exploratory Data Analysis

In [ ]:
data_sample = '../datasets/sample_sentiment_data.csv'
data_full = '../datasets/sentiment_data.csv'
data_path = data_sample if os.path.exists(data_sample) else data_full

df = pd.read_csv(data_path)
print(f'Loaded {len(df):,} reviews from {data_path}')
print('\nClass Balance:\n', df['sentiment'].value_counts())
df.head(3)

## 2. Text Normalization Pipeline
We apply HTML tag stripping, lowercasing, regex URL/punctuation removal, stopword filtering, and WordNet lemmatization.

In [ ]:
df['cleaned'] = df['review'].apply(clean_text)
df = df[df['cleaned'].str.strip() != ''].dropna(subset=['cleaned', 'sentiment'])
print(f'Retained {len(df):,} cleaned reviews.')

## 3. Train / Test Split & TF-IDF Vectorization

In [ ]:
X = df['cleaned']
y = df['sentiment'].str.lower()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print('TF-IDF Matrix Shape:', X_train_vec.shape)

## 4. Model Training & Evaluation

In [ ]:
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc * 100:.2f}%\n')
print(classification_report(y_test, y_pred))

## 5. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['positive', 'negative'])
plt.figure(figsize=(6, 5), dpi=120)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Positive', 'Negative'], yticklabels=['Positive', 'Negative'])
plt.title(f'IMDb Sentiment Classifier (Accuracy: {acc*100:.1f}%)')
plt.xlabel('Predicted')
plt.ylabel('Ground Truth')
plt.show()

## 6. Model Interpretability: Top Predictive Features

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = model.coef_[0]
top_pos = feature_names[np.argsort(coefs)[-10:]][::-1]
top_neg = feature_names[np.argsort(coefs)[:10]]

print('Top 10 Positive Predictive Words:', top_pos)
print('Top 10 Negative Predictive Words:', top_neg)